In [2]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import f1_score


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
CLASS_ORDER = ["NORMAL", "ACCIDENT", "OTHER_DISRUPTION"]

DATA_DIR = Path(".")
CANONICAL_PATH = DATA_DIR / "canonical_augmented.csv"
GROUPED_MANIFEST_PATH = DATA_DIR / "grouped_random_split_manifest_revised.csv"
SPATIAL_MANIFEST_PATH = DATA_DIR / "spatial_split_manifest_revised.csv"
OUTPUT_DIR = DATA_DIR / "lstm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


for p in [CANONICAL_PATH, GROUPED_MANIFEST_PATH, SPATIAL_MANIFEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p.resolve()}")

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: E:\26T2\5925\classification_data\lstm_outputs


In [3]:
canonical = pd.read_csv(CANONICAL_PATH)
grouped_manifest = pd.read_csv(GROUPED_MANIFEST_PATH)
spatial_manifest = pd.read_csv(SPATIAL_MANIFEST_PATH)

print("Canonical shape:", canonical.shape)
print("Grouped manifest shape:", grouped_manifest.shape)
print("Spatial manifest shape:", spatial_manifest.shape)
print("\nClass distribution:")
display(canonical["target_class"].value_counts().reindex(CLASS_ORDER).to_frame("count"))

# Basic Validation
assert canonical["event_sample_id"].is_unique, "Duplicate IDs in canonical"
assert grouped_manifest["event_sample_id"].is_unique
assert spatial_manifest["event_sample_id"].is_unique
assert set(canonical["event_sample_id"]) == set(grouped_manifest["event_sample_id"])
assert set(canonical["event_sample_id"]) == set(spatial_manifest["event_sample_id"])
assert set(grouped_manifest["split"].unique()) == {"train", "validation", "test"}
assert set(spatial_manifest["split"].unique()) == {"train", "validation", "test"}
print("\nAll basic integrity checks passed.")

Canonical shape: (5265, 90)
Grouped manifest shape: (5265, 5)
Spatial manifest shape: (5265, 5)

Class distribution:


,count
target_class,
NORMAL,3581
ACCIDENT,505
OTHER_DISRUPTION,1179



All basic integrity checks passed.


In [4]:
def build_sequence_data(
    df: pd.DataFrame,
    feature_level: str = 'A'  # 'A', 'B', 'C', 'D'
) -> np.ndarray:
    """
    Construct sequence features (N, 7, C)

    - Flow channel (2): log1p(flow), relative change (always included)

    - Time channel (4): hour_sin, hour_cos, is_weekend, is_holiday

    - Road channel (5): road_functional_hierarchy, lane_count, road_classification_type (encoding),device_type (encoding), quality_rating

    - Weather channel (6): precipitation, weather_code, apparent_temperature,temperature_2m, wind_gusts_10m, relative_humidity

    """
    all_cols = df.columns.tolist()
    # Find all flow_ columns
    flow_cols = [c for c in all_cols if c.startswith('flow_')]
    
    offset_suffixes = ['t_minus_3', 't_minus_2', 't_minus_1', 
                       't0', 't_plus_1', 't_plus_2', 't_plus_3']
    
    if 'hour_sin_anchor_time' in all_cols:
        time_suffixes = ['t_minus_3', 't_minus_2', 't_minus_1', 'anchor_time', 't_plus_1', 't_plus_2', 't_plus_3']
    else:
        time_suffixes = offset_suffixes   
          
   # 1. Traffic channel (always included)
    flow = df[[f'flow_{s}' for s in offset_suffixes]].astype(np.float32).to_numpy()  # (N, 7)
    
    # Calculate log1p(flow) and relative change
    log_flow = np.log1p(flow)
    pre_base = np.median(flow[:, :3], axis=1, keepdims=True)
    denom = pre_base + 1.0
    rel_change = np.clip((flow - pre_base) / denom, -5, 5)
    seq_channels = [np.stack([log_flow, rel_change], axis=-1)]  # (N, 7, 2)
    
      # 2. Time Channel (B)
    if feature_level in ['B', 'C', 'D']:
        time_cols = ['hour_sin', 'hour_cos', 'is_weekend', 'is_holiday']
        time_seq = []
        for suffix in time_suffixes:
            cols = [f'{col}_{suffix}' for col in time_cols]
            time_seq.append(df[cols].astype(np.float32).to_numpy()[:, np.newaxis, :])
        time_seq = np.concatenate(time_seq, axis=1)  # (N, 7, 4)
        seq_channels.append(time_seq)
    
    # 3. Roadway (C) – Static feature, repeated at each time step
    if feature_level in ['C', 'D']:
        road_cols = ['lane_count', 'road_functional_hierarchy', 'quality_rating']
   
        road_df = df[road_cols].astype(np.float32).to_numpy()  # (N, 3)
        # Expand to (N, 7, 3)
        road_seq = np.repeat(road_df[:, np.newaxis, :], 7, axis=1)
        seq_channels.append(road_seq)
    
    # 4. Weather Channel (D) – Hourly Changes
    if feature_level == 'D':
        weather_cols = ['precipitation', 'weather_code', 'apparent_temperature',
                        'temperature_2m', 'wind_gusts_10m', 'relative_humidity']
        weather_seq = []
        for suffix in time_suffixes:
            cols = [f'{col}_{suffix}' for col in weather_cols]
            weather_seq.append(df[cols].astype(np.float32).to_numpy()[:, np.newaxis, :])
        weather_seq = np.concatenate(weather_seq, axis=1)  # (N, 7, 6)
        seq_channels.append(weather_seq)
        
         # splice all channels
    X_seq = np.concatenate(seq_channels, axis=-1)  # (N, 7, C)
    return X_seq

In [5]:
def merge_split(canonical_df, manifest):
    manifest_small = manifest[["event_sample_id", "target_class", "split"]].copy()
    merged = canonical_df.merge(
        manifest_small,
        on="event_sample_id",
        how="inner",
        suffixes=("", "_manifest"),
        validate="one_to_one",
    )
    assert len(merged) == len(canonical_df)
    assert (merged["target_class"] == merged["target_class_manifest"]).all()
    return merged.drop(columns="target_class_manifest")

In [6]:
def build_lstm_model(input_shape):
    model = keras.Sequential([
        layers.LSTM(32,input_shape=input_shape, dropout=0.2, return_sequences=False),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(3, activation='softmax')
    ])
    return model

In [7]:
from sklearn.metrics import precision_recall_fscore_support
def metric_row(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "macro_precision": p,
        "macro_recall": r,
    }

In [8]:
class MacroF1Callback(Callback):
    """Calculate the Macro-F1 score for the validation set at the end of each epoch and save the history."""
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data
        self.val_macro_f1 = []

    def on_epoch_end(self, epoch, logs=None):
        X_val, y_val = self.validation_data
        y_pred = np.argmax(self.model.predict(X_val, verbose=0), axis=1)
        macro_f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
        self.val_macro_f1.append(macro_f1)
        logs['val_macro_f1'] = macro_f1  # Keras record this value for early stopping.

def run_lstm_experiment(
    split_name: str,
    manifest: pd.DataFrame,
    feature_level: str = 'A',
    seed: int = 42,
    epochs: int = 35,
    batch_size: int = 64,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 6
):
    # --- Set random seed ---
    np.random.seed(seed)
    import tensorflow as tf
    tf.random.set_seed(seed)
    import random
    random.seed(seed)
    
    # --- Data merging and partitioning---
    data = merge_split(canonical, manifest)
    seq_all = build_sequence_data(data, feature_level=feature_level)
    y_all = data["target_class"].astype(str)
    
    masks = {part: data["split"].eq(part).to_numpy() for part in ["train", "validation", "test"]}
    X_train_raw = seq_all[masks["train"]]
    y_train_raw = y_all[masks["train"]]
    X_val_raw   = seq_all[masks["validation"]]
    y_val_raw   = y_all[masks["validation"]]
    X_test_raw  = seq_all[masks["test"]]
    y_test_raw  = y_all[masks["test"]]
    
    label_map = {'NORMAL': 0, 'ACCIDENT': 1, 'OTHER_DISRUPTION': 2}
    y_train = y_train_raw.map(label_map).values.astype(np.int32)
    y_val   = y_val_raw.map(label_map).values.astype(np.int32)
    y_test  = y_test_raw.map(label_map).values.astype(np.int32)
    
    # standardization 
    mean = X_train_raw.mean(axis=(0, 1), keepdims=True)
    std = X_train_raw.std(axis=(0, 1), keepdims=True) + 1e-8
    X_train = (X_train_raw - mean) / std
    X_val   = (X_val_raw   - mean) / std
    X_test  = (X_test_raw  - mean) / std
    
    #  Category weight 
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(y_train)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, class_weights))
    
    # Building a model (dynamically inputting shapes)
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = build_lstm_model(input_shape)  
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Custom callbacks (recording verification Macro-F1) and early stop (based on val_macro_f1)
    macro_cb = MacroF1Callback(validation_data=(X_val, y_val))
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_macro_f1',  # Change to monitoring Macro-F1
        mode='max',
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )
    
    #  train 
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=[macro_cb, early_stop],
        verbose=1
    )
    
    # Predict and calculate various indicators
    # Predict using the trained model
    y_pred_train = np.argmax(model.predict(X_train, verbose=0), axis=1)
    y_pred_val   = np.argmax(model.predict(X_val,   verbose=0), axis=1)
    y_pred_test  = np.argmax(model.predict(X_test,  verbose=0), axis=1)
    
    # Convert to string for easier display and saving.
    inv_label_map = {0: 'NORMAL', 1: 'ACCIDENT', 2: 'OTHER_DISRUPTION'}
    def to_str(arr):
        return np.vectorize(inv_label_map.__getitem__)(arr)
    y_train_str = to_str(y_train)
    y_val_str   = to_str(y_val)
    y_test_str  = to_str(y_test)
    y_pred_train_str = to_str(y_pred_train)
    y_pred_val_str   = to_str(y_pred_val)
    y_pred_test_str  = to_str(y_pred_test)
    
    results = []
    for part, y_true, y_pred in [
        ("train", y_train_str, y_pred_train_str),
        ("validation", y_val_str, y_pred_val_str),
        ("test", y_test_str, y_pred_test_str)
    ]:
        metrics = metric_row(y_true, y_pred)
        row = {
            "split_scheme": split_name,
            "seed": seed,
            "feature_level": feature_level,
            "partition": part,
            "accuracy": metrics["accuracy"],
            "macro_precision": metrics.get("macro_precision", np.nan),
            "macro_recall": metrics.get("macro_recall", np.nan),
            "macro_f1": metrics["macro_f1"],
            "balanced_accuracy": metrics.get("balanced_accuracy", np.nan),
        }
        results.append(row)
    results_df = pd.DataFrame(results)
    
    # Add best validation Macro-F1
    best_val_macro_f1 = max(macro_cb.val_macro_f1) if macro_cb.val_macro_f1 else np.nan
    results_df.loc[results_df["partition"] == "validation", "best_val_macro_f1"] = best_val_macro_f1
    
    
    # Indicators for each category
    per_class_list = []
    for part, y_true, y_pred in [
        ("train", y_train_str, y_pred_train_str),
        ("validation", y_val_str, y_pred_val_str),
        ("test", y_test_str, y_pred_test_str)
    ]:
        report = classification_report(
            y_true, y_pred,
            labels=CLASS_ORDER,
            output_dict=True,
            zero_division=0
        )
        for class_name in CLASS_ORDER:
            per_class_list.append({
                "split_scheme": split_name,
                "seed": seed,
                "feature_level": feature_level,
                "partition": part,
                "class": class_name,
                "precision": report[class_name]["precision"],
                "recall": report[class_name]["recall"],
                "f1_score": report[class_name]["f1-score"],
                "support": report[class_name]["support"]
            })
    per_class_df = pd.DataFrame(per_class_list)
    
    
    # Prediction results
    
    ids_train = data.loc[masks["train"], "event_sample_id"].to_numpy()
    ids_val   = data.loc[masks["validation"], "event_sample_id"].to_numpy()
    ids_test  = data.loc[masks["test"], "event_sample_id"].to_numpy()
    
    pred_list = []
    for part, ids, y_true, y_pred in [
        ("train", ids_train, y_train_str, y_pred_train_str),
        ("validation", ids_val, y_val_str, y_pred_val_str),
        ("test", ids_test, y_test_str, y_pred_test_str)
    ]:
        pred_list.append(pd.DataFrame({
            "event_sample_id": ids,
            "split_scheme": split_name,
            "seed": seed,
            "feature_level": feature_level,
            "partition": part,
            "y_true": y_true,
            "y_pred": y_pred,
        }))
    prediction_df = pd.concat(pred_list, ignore_index=True)
 
 
    return {
        "metrics": results_df,
        "per_class": per_class_df,
        "predictions": prediction_df
    }

In [9]:
# Define the experimental range
seeds = [42, 52, 62]
levels = ['A', 'B', 'C', 'D']
splits = [("Grouped-random", grouped_manifest), ("Spatial", spatial_manifest)]

all_metrics = []
all_per_class = []
all_predictions = []

for split_name, manifest in splits:
    for level in levels:
        for seed in seeds:
            print(f"\n Running {split_name} | Level {level} | Seed {seed}")
            out = run_lstm_experiment(
                split_name=split_name,
                manifest=manifest,
                feature_level=level,
                seed=seed
            )
            all_metrics.append(out["metrics"])
            all_per_class.append(out["per_class"])
            all_predictions.append(out["predictions"])

# merge
df_all_metrics = pd.concat(all_metrics, ignore_index=True)
df_all_per_class = pd.concat(all_per_class, ignore_index=True)
# df_all_predictions = pd.concat(all_predictions, ignore_index=True)  


 Running Grouped-random | Level A | Seed 42
Epoch 1/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4380 - loss: 1.1016 - val_accuracy: 0.3189 - val_loss: 1.0966 - val_macro_f1: 0.2782
Epoch 2/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3490 - loss: 1.0996 - val_accuracy: 0.3036 - val_loss: 1.1028 - val_macro_f1: 0.2742
Epoch 3/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3493 - loss: 1.0970 - val_accuracy: 0.3087 - val_loss: 1.1016 - val_macro_f1: 0.2745
Epoch 4/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3536 - loss: 1.0980 - val_accuracy: 0.2730 - val_loss: 1.1031 - val_macro_f1: 0.2492
Epoch 5/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3674 - loss: 1.0996 - val_accuracy: 0.3036 - val_loss: 1.1021 - val_macro_f1: 0.2699
Epoch 6/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3645 - loss: 1.0984 - val_accuracy: 0.3189 - val_loss: 1.0949 - val_macro_f1: 0.2752
Epoch 7/35
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy

In [11]:
OUTPUT_DIR = Path("lstm_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# all_metrics_per_seed.csv

df_all_metrics.to_csv(OUTPUT_DIR / "lstm_all_metrics_per_seed.csv", index=False)
print("Saved lstm_all_metrics_per_seed.csv")


# all_per_class_metrics.csv
df_all_per_class.to_csv(OUTPUT_DIR / "lstm_all_per_class_metrics.csv", index=False)
print("Saved lstm_all_per_class_metrics.csv")


# final_per_class_summary.csv
# Summarize by split_scheme, feature_level, and class (mean and standard deviation)
per_class_summary = df_all_per_class.groupby(
    ["split_scheme", "feature_level", "class", "partition"]
).agg({
    "precision": ["mean", "std"],
    "recall": ["mean", "std"],
    "f1_score": ["mean", "std"],
    "support": ["mean"]  
}).reset_index()
# Flattened Listings
per_class_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in per_class_summary.columns.values]
per_class_summary.to_csv(OUTPUT_DIR / "lstm_final_per_class_summary.csv", index=False)
print("Saved lstm_final_per_class_summary.csv")

# final_test_summary.csv
# Test set only, summarizing macro_f1, accuracy, etc. by split_scheme, feature_level

test_metrics = df_all_metrics[df_all_metrics["partition"] == "test"]
test_summary = test_metrics.groupby(
    ["split_scheme", "feature_level"]
).agg({
    "accuracy": ["mean", "std"],
    "macro_f1": ["mean", "std"],
    "macro_precision": ["mean", "std"],
    "macro_recall": ["mean", "std"],
    "balanced_accuracy": ["mean", "std"]
}).reset_index()
test_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in test_summary.columns.values]
test_summary.to_csv(OUTPUT_DIR / "lstm_final_test_summary.csv", index=False)
print("Saved lstm_final_test_summary.csv")

 
# spatial_generalisation_gap.csv
# Spatial Macro-F1 - Grouped Random Macro-F1

# filter the test set
test_only = df_all_metrics[df_all_metrics["partition"] == "test"]

pivot = test_only.pivot_table(
    index=["feature_level", "seed"],
    columns="split_scheme",
    values="macro_f1"
).reset_index()
# Calculate the difference（Spatial - Grouped-random）
if "Spatial" in pivot.columns and "Grouped-random" in pivot.columns:
    pivot["gap"] = pivot["Spatial"] - pivot["Grouped-random"]
    # Sum of mean and standard deviation by level
    gap_summary = pivot.groupby("feature_level").agg({
        "gap": ["mean", "std"]
    }).reset_index()
    gap_summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in gap_summary.columns.values]
    gap_summary.to_csv(OUTPUT_DIR / "lstm_spatial_generalisation_gap.csv", index=False)
    print("Saved lstm_spatial_generalisation_gap.csv")
else:
    print("Cannot compute gap: missing 'Spatial' or 'Grouped-random' in splits.")

Saved lstm_all_metrics_per_seed.csv
Saved lstm_all_per_class_metrics.csv
Saved lstm_final_per_class_summary.csv
Saved lstm_final_test_summary.csv
Saved lstm_spatial_generalisation_gap.csv


In [ ]:
# print("\n" + "="*60)
# print("Running Grouped-random experiment")
# print("="*60)
# grouped_results = run_lstm_experiment("Grouped-random", grouped_manifest)

# print("\n" + "="*60)
# print("Running Spatial experiment")
# print("="*60)
# spatial_results = run_lstm_experiment("Spatial", spatial_manifest)

In [ ]:
# # Summary of Macro-F1 comparisons on test sets
# summary = pd.DataFrame({
#     "Split": ["Grouped-random", "Spatial"],
#     "Test Macro-F1": [
#         grouped_results["metrics"].loc[grouped_results["metrics"]["partition"]=="test", "macro_f1"].values[0],
#         spatial_results["metrics"].loc[spatial_results["metrics"]["partition"]=="test", "macro_f1"].values[0]
#     ]
# })
# display(summary)

In [ ]:
# =============================================
# 实验 A: Traffic only
# =============================================
print("\n" + "="*60)
print("Running Grouped-random experiment - Feature Level A")
print("="*60)
results_A_grouped = run_lstm_experiment("Grouped-random", grouped_manifest, feature_level='A')

print("\n" + "="*60)
print("Running Spatial experiment - Feature Level A")
print("="*60)
results_A_spatial = run_lstm_experiment("Spatial", spatial_manifest, feature_level='A')

In [ ]:
# =============================================
# 实验 B: Traffic + Time
# =============================================
print("\n" + "="*60)
print("Running Grouped-random experiment - Feature Level B")
print("="*60)
results_B_grouped = run_lstm_experiment("Grouped-random", grouped_manifest, feature_level='B')

print("\n" + "="*60)
print("Running Spatial experiment - Feature Level B")
print("="*60)
results_B_spatial = run_lstm_experiment("Spatial", spatial_manifest, feature_level='B')

In [ ]:
# =============================================
# 实验 C: Traffic + Time + Road
# =============================================
print("\n" + "="*60)
print("Running Grouped-random experiment - Feature Level C")
print("="*60)
results_C_grouped = run_lstm_experiment("Grouped-random", grouped_manifest, feature_level='C')

print("\n" + "="*60)
print("Running Spatial experiment - Feature Level C")
print("="*60)
results_C_spatial = run_lstm_experiment("Spatial", spatial_manifest, feature_level='C')

In [ ]:
# =============================================
# 实验 D: Traffic + Time + Road + Weather
# =============================================
print("\n" + "="*60)
print("Running Grouped-random experiment - Feature Level D")
print("="*60)
results_D_grouped = run_lstm_experiment("Grouped-random", grouped_manifest, feature_level='D')

print("\n" + "="*60)
print("Running Spatial experiment - Feature Level D")
print("="*60)
results_D_spatial = run_lstm_experiment("Spatial", spatial_manifest, feature_level='D')

In [ ]:
summary = pd.DataFrame({
    "Feature Level": ["A", "B", "C", "D"],
    "Grouped-random Test Macro-F1": [
        results_A_grouped["metrics"].loc[results_A_grouped["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_B_grouped["metrics"].loc[results_B_grouped["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_C_grouped["metrics"].loc[results_C_grouped["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_D_grouped["metrics"].loc[results_D_grouped["metrics"]["partition"]=="test", "macro_f1"].values[0],
    ],
    "Spatial Test Macro-F1": [
        results_A_spatial["metrics"].loc[results_A_spatial["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_B_spatial["metrics"].loc[results_B_spatial["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_C_spatial["metrics"].loc[results_C_spatial["metrics"]["partition"]=="test", "macro_f1"].values[0],
        results_D_spatial["metrics"].loc[results_D_spatial["metrics"]["partition"]=="test", "macro_f1"].values[0],
    ]
})
display(summary)